In [1]:
import pandas as pd
import numpy as np

# 1. Carregar a base original (Morumbi)

In [2]:
# Carrega o arquivo de entrada (arquivo organizado em `data/filling_Ceps/`)
df = pd.read_csv('../../data/filling_Ceps/Elvira Brandão Morumbi - Euvira Brandão Dados ADS_coords_corrigidas_com_enderecos.csv')

# 2. Tratar a renda para número

In [3]:
import re

def parse_renda_seguro(x):
    if pd.isna(x):
        return np.nan
    s = str(x)
    s = re.sub(r"[^0-9,\.]", "", s)
    if "," in s:
        s = s.replace(".", "")
    elif s.count(".") > 1:
        s = s.replace(".", "")
    s = s.replace(",", ".")
    try:
        return float(s)
    except ValueError:
        return np.nan


df["renda_media_num"] = df["renda_media"].apply(parse_renda_seguro)


# 3. Criar Total_0_9 (0–4 + 5–9 anos)

In [4]:
df["Total_0_9"] = (
    df["v01031_0_4anos"].fillna(0) +
    df["v01032_5_9anos"].fillna(0)
)

# 4. Corrigir renda para 2025 (inflação)

In [5]:
inflation_factor = 1.155
df["renda_atualizada_2025"] = df["renda_media_num"] * inflation_factor

# 5. Score linha a linha (informativo, não consolidado)

In [6]:
df["score_trafego_2025"] = df["renda_atualizada_2025"] * df["Total_0_9"]

# 6. Consolidar por CEP (SEM SOMAR — usar média!)

In [7]:
df_cep = (
    df.groupby("CEP", as_index=False)
      .agg({
          "Bairro": lambda x: x.mode().iat[0] if not x.mode().empty else x.iloc[0],
          "renda_atualizada_2025": "median",
          "Total_0_9": "median",
          "populacao_total": "median"
      })
)

In [8]:
# Renomear colunas para refletir que são médias, não totais
df_cep = df_cep.rename(columns={
    "renda_atualizada_2025": "renda_mediana_2025",
    "Total_0_9": "mediana_criancas_0_9",
    "populacao_total": "populacao_mediana"
})

# 7. Score final no nível do CEP (correto)

In [9]:
df_cep["score_trafego_2025"] = (
    df_cep["renda_mediana_2025"] * df_cep["mediana_criancas_0_9"]
)

# 8. Ranking final

In [10]:
top_ceps = df_cep.sort_values("score_trafego_2025", ascending=False)

print(top_ceps.head(15))

           CEP                            Bairro  renda_mediana_2025  \
512  05635-050                Jardim Monte Kemel         24449.50200   
103  04719-905  Chácara Santo Antônio (Zona Sul)         32647.68045   
667  05709-040                       Vila Suzana         36272.94825   
134  04729-060                  Jardim Caravelas         19448.42130   
42   04583-909                     Vila Cordeiro         28934.10135   
118  04726-160                     Vila Cruzeiro         24868.25880   
617  05679-050           Jardim Panorama D'Oeste         35331.79650   
501  05634-001                Jardim Monte Kemel         22594.04070   
725  05726-140                      Vila Andrade         19572.22575   
16   04567-002                    Cidade Monções         27318.29100   
136  04730-000                   Várzea de Baixo         18074.72205   
664  05707-400                 Parque do Morumbi         21891.88155   
67   04709-901                       Santo Amaro         25861.4

In [11]:
# Arredondar para 2 casas decimais (padrão monetário)
top_ceps["renda_mediana_2025"] = top_ceps["renda_mediana_2025"].round(2)
top_ceps["score_trafego_2025"] = top_ceps["score_trafego_2025"].round(2)

In [12]:
# Salvar resultado agregado na pasta do notebook (comportamento original)
top_ceps.to_csv("morumbi_top_ceps_2025.csv", index=False)

In [13]:
# Lista dos bairros desejados
bairros_desejados = ["Vila Sônia", "Ferreira", "Paraisópolis", "Jardim Maria Duarte", "Vila Andrade"]

# Filtrar diretamente no top_ceps
top_ceps_filtrado = top_ceps[top_ceps["Bairro"].isin(bairros_desejados)]

top_ceps_filtrado

,CEP,Bairro,renda_mediana_2025,mediana_criancas_0_9,populacao_mediana,score_trafego_2025
725,05726-140,Vila Andrade,19572.23,109.0,612.0,2133372.61
705,05717-250,Vila Andrade,24980.56,70.0,436.0,1748639.16
717,05724-005,Vila Andrade,17073.24,98.5,619.5,1681714.60
730,05727-230,Vila Andrade,18435.43,71.0,519.0,1308915.84
700,05717-180,Vila Andrade,36600.09,31.0,285.0,1134602.80
...,...,...,...,...,...,...
590,05663-030,Paraisópolis,2226.70,19.0,237.0,42307.33
222,05524-000,Ferreira,3780.83,10.0,148.0,37808.35
574,05659-900,Paraisópolis,10101.49,0.0,45.0,0.00
721,05725-060,Vila Andrade,8422.50,0.0,120.0,0.00


In [14]:
top_ceps_filtrado.to_csv("morumbi_top_ceps_filtrados_2025.csv", index=False)